# Day 035 Solution — Auto-Analyst Capstone

End-to-end pipeline: fetch text → extract structured info (Pydantic + JSON mode) → batch async → generate digest.
All sources are inline text strings for gate compatibility.

In [ ]:
import requests
from pathlib import Path

def fetch_text(source: str) -> dict:
    src  = str(source)
    text = None
    kind = 'text'

    if src.startswith('http://') or src.startswith('https://'):
        try:
            response = requests.get(src, timeout=10)
            response.raise_for_status()
            text = response.text
            kind = 'url'
        except Exception as e:
            text = '[fetch error: ' + str(e) + ']'
            kind = 'url_error'
    else:
        try:
            p = Path(src)
            if p.exists() and p.is_file():
                text = p.read_text(encoding='utf-8')
                kind = 'file'
        except Exception:
            pass

    if text is None:
        text = src
        kind = 'text'

    return {
        'source':     src,
        'kind':       kind,
        'content':    text,
        'char_count': len(text),
    }


import json
import ollama
from pydantic import BaseModel, Field

class ArticleInfo(BaseModel):
    title:      str       = Field(description='Topic or title in 3-6 words')
    summary:    str       = Field(description='One sentence summary')
    sentiment:  str       = Field(description='positive, negative, or neutral')
    key_points: list[str] = Field(default_factory=list,
                                  description='Up to 3 key points as short phrases')

def extract_info(doc: dict, model: str = 'llama3.2') -> dict:
    schema = ArticleInfo.model_json_schema()
    prompt = (
        'Extract information from the document below. '
        'Return valid JSON matching this schema:\n'
        + json.dumps(schema, indent=2)
        + '\n\nDocument:\n' + doc['content'][:1500]
    )
    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
        )
        info = ArticleInfo.model_validate_json(response['message']['content'])
        return {**doc, 'info': info.model_dump(), 'status': 'ok',    'error': None}
    except Exception as e:
        return {**doc, 'info': None,               'status': 'error', 'error': str(e)}


import asyncio
import json
import ollama

async def async_extract(doc: dict, model: str = 'llama3.2') -> dict:
    client = ollama.AsyncClient()
    schema = ArticleInfo.model_json_schema()
    prompt = (
        'Extract information from the document below. '
        'Return valid JSON matching this schema:\n'
        + json.dumps(schema, indent=2)
        + '\n\nDocument:\n' + doc['content'][:1500]
    )
    try:
        response = await client.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
        )
        info = ArticleInfo.model_validate_json(response['message']['content'])
        return {**doc, 'info': info.model_dump(), 'status': 'ok',    'error': None}
    except Exception as e:
        return {**doc, 'info': None,               'status': 'error', 'error': str(e)}


async def batch_extract(docs: list, max_concurrent: int = 3,
                        model: str = 'llama3.2') -> list[dict]:
    if not docs:
        return []
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(doc):
        async with sem:
            return await async_extract(doc, model)
    return list(await asyncio.gather(*[_run(d) for d in docs]))


import ollama

def generate_digest(results: list, model: str = 'llama3.2') -> str:
    ok     = [r for r in results if r.get('status') == 'ok']
    errors = [r for r in results if r.get('status') == 'error']
    if not ok:
        return 'No articles extracted successfully (' + str(len(errors)) + ' errors).'
    lines = [
        '=== Auto-Analyst Digest ===',
        str(len(results)) + ' sources processed: '
        + str(len(ok)) + ' ok, ' + str(len(errors)) + ' failed.\n',
    ]
    for i, r in enumerate(ok, 1):
        info      = r.get('info') or {}
        title     = info.get('title',     'Untitled')
        summary   = info.get('summary',   '')
        sentiment = info.get('sentiment', 'unknown')
        kp        = info.get('key_points', [])
        kp_text   = '; '.join(kp[:3]) if kp else ''
        lines.append('[' + str(i) + '] ' + title + '  [' + sentiment + ']')
        lines.append('    ' + summary)
        if kp_text:
            lines.append('    Key points: ' + kp_text)
        lines.append('')
    context  = '\n'.join(lines)
    prompt   = (
        context + '\n\n'
        'Write a 3-4 sentence editorial digest identifying '
        'the main themes, patterns, and key insights across all articles.'
    )
    response = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response['message']['content']


import asyncio

class DigestPipeline:
    def __init__(self, model: str = 'llama3.2', max_concurrent: int = 3):
        self.model          = model
        self.max_concurrent = max_concurrent
        self._sources: list = []

    def add_source(self, source) -> 'DigestPipeline':
        self._sources.append(source)
        return self

    async def process(self) -> dict:
        docs     = [fetch_text(s) for s in self._sources]
        results  = await batch_extract(docs, self.max_concurrent, self.model)
        digest   = generate_digest(results, self.model)
        ok_count = sum(1 for r in results if r.get('status') == 'ok')
        return {
            'source_count': len(docs),
            'ok_count':     ok_count,
            'results':      results,
            'digest':       digest,
        }

    def run(self) -> dict:
        return asyncio.run(self.process())

## Step 1 — Fetch Text Sources

In [ ]:
SOURCES = [
    'Python is a versatile high-level programming language '
    'popular in AI, data science, and web development.',

    'Machine learning is a subset of artificial intelligence '
    'that enables computers to learn patterns from data.',

    'Large language models like GPT and LLaMA are trained '
    'on vast amounts of text to generate human-like responses.',
]

docs = [fetch_text(s) for s in SOURCES]
print(f'Loaded {len(docs)} documents:')
for d in docs:
    print(f'  kind={d["kind"]!r}  chars={d["char_count"]}  '
          f'preview={d["content"][:40]!r}...')

assert len(docs) == 3
assert all(d['kind'] == 'text' for d in docs)

## Step 2 — Batch Extract (Async)

In [ ]:
results = await batch_extract(docs, max_concurrent=3)
print(f'\nExtracted {len(results)} results:')
for r in results:
    status = r['status']
    info   = r.get('info') or {}
    title  = info.get('title', 'N/A')
    sent   = info.get('sentiment', 'N/A')
    print(f'  [{status}] {title!r}  sentiment={sent!r}')

assert len(results) == 3
assert all('status' in r for r in results)

## Step 3 — Generate Digest

In [ ]:
digest = generate_digest(results)
print('\n' + digest)

assert isinstance(digest, str) and len(digest.strip()) >= 50

## Step 4 — DigestPipeline Full Run

In [ ]:
pipeline = (
    DigestPipeline(model='llama3.2', max_concurrent=3)
    .add_source('Renewable energy sources like solar and wind '
                'are becoming increasingly cost-competitive.')
    .add_source('Quantum computing promises exponential speedups '
                'for certain classes of problems over classical computers.')
)

output = await pipeline.process()
print(f'source_count : {output["source_count"]}')
print(f'ok_count     : {output["ok_count"]}')
print(f'digest       : {output["digest"][:120]}...')

assert output['source_count'] == 2
assert isinstance(output['digest'], str) and output['digest'].strip()
assert 'results' in output and len(output['results']) == 2

print('\nAuto-Analyst complete!')